In [ ]:
!pip install -q unsloth gradio

In [1]:
import os, sys, random
import unsloth                                  # must be imported before transformers
import pandas as pd

LINGOQA      = "/kaggle/input/datasets/hamsaa123/lingoqa"
ADAPTER_DIR  = "/kaggle/input/datasets/hamsaa123/lora-cues/lora_cues"
FEATURES_CSV = "/kaggle/input/datasets/hamsaa123/lora-cues/ego_features.csv"
CUES_JSON    = "/kaggle/input/datasets/hamsaa123/cues-all/cues_all.json"
CODE_ROOT    = "/kaggle/input/datasets/hamsaa123/perceptqa-code"

# the zip unpacks either as perceptqa-code/perceptqa or perceptqa-code/repo/perceptqa
CODE_DIR = next(d for d in [CODE_ROOT, f"{CODE_ROOT}/repo"] if os.path.isdir(f"{d}/perceptqa"))

for name, path in {"lingoqa": LINGOQA, "adapter": ADAPTER_DIR, "features": FEATURES_CSV,
                   "cues": CUES_JSON, "code": CODE_DIR}.items():
    assert os.path.exists(path), f"missing {name}: {path}"
    print(f"{name:9s} {path}")

sys.path.insert(0, CODE_DIR)
from perceptqa import PerceptQA
from perceptqa.cues.ego import EgoEstimator, mine_label

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
lingoqa   /kaggle/input/datasets/hamsaa123/lingoqa
adapter   /kaggle/input/datasets/hamsaa123/lora-cues/lora_cues
features  /kaggle/input/datasets/hamsaa123/lora-cues/ego_features.csv
cues      /kaggle/input/datasets/hamsaa123/cues-all/cues_all.json
code      /kaggle/input/datasets/hamsaa123/perceptqa-code/repo


In [2]:
def parquet(split):
    """Path of a LingoQA table (evaluation / action)."""
    for p in [f"{LINGOQA}/{split}.parquet", f"{LINGOQA}/{split}/{split}.parquet"]:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(split)

ev = pd.read_parquet(parquet("evaluation"))      # official test set: 500 questions, 2 human answers each
ac = pd.read_parquet(parquet("action"))          # training set: used only to train the motion classifier
ev["question_id"] = ev["question_id"].astype(str)
EVAL_FRAMES = f"{LINGOQA}/evaluation/evaluation"
print(f"evaluation rows: {len(ev)} | action rows: {len(ac)}")

evaluation rows: 1000 | action rows: 129903


In [3]:
# Motion labels are read from the training answers ("slowing down", "stopped", ...). Test clips are excluded.
features = pd.read_csv(FEATURES_CSV).drop_duplicates("segment_id").set_index("segment_id")

ac["motion"] = [mine_label(q, a) for q, a in zip(ac["question"], ac["answer"])]
labels = (ac.dropna(subset=["motion"])
            .groupby("segment_id")["motion"]
            .agg(lambda s: s.iloc[0] if s.nunique() == 1 else None)   # keep clips whose answers agree
            .dropna())
labels = labels[~labels.index.isin(set(ev["segment_id"]))]            # never train on test clips
labels = labels.groupby(labels).head(600)                             # balance the classes

ego = EgoEstimator().fit(features, labels)
ego.save("/kaggle/working/ego_model.joblib")
print("accuracy per motion field (%):", ego.gates)

accuracy per motion field (%): {'stop': np.float64(84.9), 'trend': np.float64(86.7), 'turn': np.float64(69.2)}


In [5]:
# Find the folder that actually contains the adapter files inside the lora-cues dataset
LORA_ROOT = "/kaggle/input/datasets/hamsaa123/lora-cues"
hits = [root for root, dirs, files in os.walk(LORA_ROOT) if "adapter_config.json" in files]

if hits:
    ADAPTER_DIR = hits[0]
    print("adapter found:", ADAPTER_DIR)
    print("files        :", sorted(os.listdir(ADAPTER_DIR)))
else:
    print("no adapter_config.json in the dataset. Everything it contains:")
    for root, dirs, files in os.walk(LORA_ROOT):
        print(" ", root, "->", files[:10])

adapter found: /kaggle/input/datasets/hamsaa123/lora-cues/lora_cues/lora_cues
files        : ['README.md', 'adapter_config.json', 'adapter_model.safetensors']


In [6]:
# question router + cue extractors + prompt builder + Qwen3.5-2B with the fine-tuned adapter
qa = PerceptQA(adapter=ADAPTER_DIR,
               ego_model_path="/kaggle/working/ego_model.joblib",
               use_cues=True)

==((====))==  Unsloth 2026.11.5: Fast Qwen3_5 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.
Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

Unsloth: the model has bfloat16 layers on GPU 0 (Tesla T4), which has no native bfloat16, so they run slowly. Load with dtype = torch.float16, or keep the model on bfloat16 GPUs with CUDA_VISIBLE_DEVICES.


In [7]:
# official questions, each with its two human answers
questions = (ev.groupby("question_id")
               .agg(question=("question", "first"), segment_id=("segment_id", "first"), refs=("answer", list))
               .reset_index().to_dict("records"))
random.Random(0).shuffle(questions)

q = questions[0]
out = qa(f"{EVAL_FRAMES}/{q['segment_id']}", q["question"])

print("question      :", q["question"])
print("question type :", out["question_type"])
print("cues used     :", out["cues_used"])
print("motion        :", out["motion"])
print("answer        :", out["answer"])
print("human answers :", q["refs"])
print("time          :", out["seconds"], "s")
for name, img in out["zooms"]:
    print("zoom:", name)
    display(img)

`causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
`causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


question      : How many parked cars car you spot?
question type : counting
cues used     : ['TIME']
motion        : slowing down
answer        : I spot no parked cars.
human answers : ['One.', '1']
time          : 131.9 s


In [8]:
import gradio as gr
from PIL import Image

EXAMPLES = {f"{q['question'][:70]}  [{q['segment_id'][:8]}]": q for q in questions[:80]}

def answer(choice, own_question):
    """Run the pipeline on the chosen clip and return everything the app shows."""
    q = EXAMPLES[choice]
    clip = f"{EVAL_FRAMES}/{q['segment_id']}"
    question = own_question.strip() or q["question"]
    out = qa(clip, question)

    frames = [(Image.open(f"{clip}/{i}.jpg"), f"t = {i - 4} s") for i in range(5)]
    zooms = [(img, name) for name, img in out["zooms"]]
    trace = (f"question type : {out['question_type']}\n"
             f"cues used     : {', '.join(out['cues_used']) or 'none'}\n"
             f"motion        : {out['motion'] or 'not stated'}\n"
             f"time          : {out['seconds']} s")
    refs = "\n".join(f"- {r}" for r in q["refs"]) if not own_question.strip() else "-"
    return frames, zooms, out["answer"], trace, refs

with gr.Blocks(title="PerceptQA") as app:
    gr.Markdown("## PerceptQA: driving question answering with classical-CV cues and a fine-tuned 2B VLM")
    with gr.Row():
        choice = gr.Dropdown(list(EXAMPLES), value=list(EXAMPLES)[0], label="Clip and question (LingoQA evaluation set)")
        own = gr.Textbox(label="Or ask your own question about this clip")
    run = gr.Button("Answer", variant="primary")
    frames = gr.Gallery(label="Input: 5 frames, 1 second apart", columns=5, height=200)
    with gr.Row():
        zooms = gr.Gallery(label="Zoom crops added by the system", columns=2, height=220)
        with gr.Column():
            ans = gr.Textbox(label="PerceptQA answer")
            trace = gr.Textbox(label="Evidence used", lines=4)
            refs = gr.Textbox(label="Human answers", lines=3)
    run.click(answer, [choice, own], [frames, zooms, ans, trace, refs])

app.launch(share=True)      # prints a public gradio.live link

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://089bca3e815ece508c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
